## Import Datas

In [ ]:
import sys
import os
from google.colab import drive
drive.mount('/content/drive')
folder = "/content/drive/MyDrive/Thèse de doctorat/Redaction - rapports et articles/Articles de conférence redigés/ICATH 2026/code/data/preprocessed options datas/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
import numpy as np
import random
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
TechnologyUnderlying = [ "AAPL", "AMZN", "GOOGL", "META", "MSFT" ]
FinanceUnderlying = [ "BAC", "C", "GS", "JPM", "WFC" ]
HealthUnderlying = [ "ABBV", "JNJ", "MRK", "PFE", "UNH" ]
AutomobileUnderlying = [ "F", "GM", "LCID", "RIVN", "TSLA" ]

listTickers = AutomobileUnderlying + FinanceUnderlying + HealthUnderlying + TechnologyUnderlying

In [ ]:
dataset = {
    "AAPL" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "AMZN" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GOOGL" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "META" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "MSFT" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "BAC" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "C" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GS" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},#
    "JPM" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "WFC" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "ABBV" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "JNJ" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "MRK" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "PFE" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "UNH" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "F" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GM" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "LCID" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "RIVN" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "TSLA" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
}

In [ ]:
for ticker in dataset.keys():
    train_path = os.path.join(folder, f"{ticker}_train.csv")
    test_path = os.path.join(folder, f"{ticker}_test.csv")

    if not os.path.exists(train_path):
        print(f"Missing: {train_path}")
        continue

    train_dataset = pd.read_csv(train_path)
    test_dataset = pd.read_csv(test_path)

    train_dataset = train_dataset.fillna(train_dataset.mean(numeric_only=True))
    test_dataset = test_dataset.fillna(test_dataset.mean(numeric_only=True))

    dataset[ticker]["train"] = train_dataset
    dataset[ticker]["test"] = test_dataset

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Define functions

In [ ]:
!pip install optuna

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
import xgboost as xgb
from xgboost import XGBRegressor

In [ ]:
import optuna
import warnings

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

In [ ]:
def train_xgboost_with_optuna(X_train, y_train, X_test, y_test, best_params):

    params = best_params.copy()
    params["objective"] = "reg:squarederror"
    params["eval_metric"] = "rmse"
    params["tree_method"] = "hist"

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=500,
        evals=[(dtrain, "train")],
        early_stopping_rounds=50,
        verbose_eval=False
    )

    preds = model.predict(dtest)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    nrmse = rmse / np.mean(y_test)

    return mae, rmse, nrmse


def objective_for_xgboost(trial, X_train, y_train):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 0, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 1),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5)
    }

    model = XGBRegressor(**params)
    score = cross_val_score(model, X_train, y_train, cv=5, scoring="neg_mean_squared_error")

    return -score.mean()

def predict_current_price_using_xgboost(option_type, ticher):
  for proxy in feature_combinations:
    train_dataset = dataset[ticker]["train"]
    train_dataset = train_dataset[train_dataset["optionType"] == option_type]
    X_train = train_dataset[list_histos_datas_inputs + proxy].values
    y_train = train_dataset[["lastPrice"]].values

    test_dataset = dataset[ticker]["test"]
    test_dataset = test_dataset[test_dataset["optionType"] == option_type]
    X_test = test_dataset[list_histos_datas_inputs+ proxy].values
    y_test = test_dataset[["lastPrice"]].values

    #Normaliser les dataset
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.fit_transform(X_test)

    study = study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(lambda trial: objective_for_xgboost(trial, X_train, y_train), n_trials=10)
    best_params = study.best_params

    mae, rmse, nrmse = train_xgboost_with_optuna(
        X_train, y_train,
        X_test, y_test,
        best_params
    )
    print(f"{proxy} for {ticker} => (MAE={mae:.3f}; RMSE={rmse:.3f}; ; NRMSE={nrmse:.3f})")


## Price options using extreme gradient boosting machine

In [ ]:
ticker = "F"

In [ ]:
test_size=0.2
random_state=42
target = 'lastPrice'
list_histos_datas_inputs = ["strike", "impliedVolatility", "timeToMaturity",
                            "riskFreeRate", "underlyingLastPrice"]

feature_combinations = [
    [],

    ["GS_" + ticker], ["SC_" + ticker], ["VIX"], ["PCR"],

    ["GS_" + ticker , "SC_" + ticker], ["GS_" + ticker , "VIX"], ["GS_" + ticker , "PCR"], ["SC_" + ticker , "VIX"], ["SC_" + ticker , "PCR"], ["VIX" , "PCR"],

    ["GS_" + ticker , "SC_" + ticker , "VIX"], ["GS_" + ticker , "SC_" + ticker , "PCR"], ["SC_" + ticker , "VIX" , "PCR"], ["SC_" + ticker , "VIX" , "PCR"],

	["GS_" + ticker , "SC_" + ticker , "VIX" , "PCR"]
]

In [ ]:
predict_current_price_using_xgboost("put", ticker)

[] for F => (MAE=0.146; RMSE=0.234; ; NRMSE=0.275)
['GS_F'] for F => (MAE=0.147; RMSE=0.228; ; NRMSE=0.268)
['SC_F'] for F => (MAE=0.147; RMSE=0.228; ; NRMSE=0.268)
['VIX'] for F => (MAE=0.147; RMSE=0.228; ; NRMSE=0.268)
['PCR'] for F => (MAE=0.147; RMSE=0.228; ; NRMSE=0.268)
['GS_F', 'SC_F'] for F => (MAE=0.149; RMSE=0.216; ; NRMSE=0.254)
['GS_F', 'VIX'] for F => (MAE=0.146; RMSE=0.214; ; NRMSE=0.252)
['GS_F', 'PCR'] for F => (MAE=0.145; RMSE=0.213; ; NRMSE=0.250)
['SC_F', 'VIX'] for F => (MAE=0.145; RMSE=0.212; ; NRMSE=0.249)
['SC_F', 'PCR'] for F => (MAE=0.146; RMSE=0.214; ; NRMSE=0.251)
['VIX', 'PCR'] for F => (MAE=0.144; RMSE=0.210; ; NRMSE=0.247)
['GS_F', 'SC_F', 'VIX'] for F => (MAE=0.143; RMSE=0.230; ; NRMSE=0.270)
['GS_F', 'SC_F', 'PCR'] for F => (MAE=0.145; RMSE=0.234; ; NRMSE=0.275)
['SC_F', 'VIX', 'PCR'] for F => (MAE=0.145; RMSE=0.234; ; NRMSE=0.275)
['SC_F', 'VIX', 'PCR'] for F => (MAE=0.145; RMSE=0.234; ; NRMSE=0.275)
['GS_F', 'SC_F', 'VIX', 'PCR'] for F => (MAE=0.132; R

In [ ]:
# [] for F => (MAE=0.146; RMSE=0.234; ; NRMSE=0.275); ['GS_F', 'SC_F', 'VIX', 'PCR'] for F => (MAE=0.132; RMSE=0.208; ; NRMSE=0.245)

In [ ]:
predict_current_price_using_xgboost("call", ticker)

[] for F => (MAE=0.146; RMSE=0.234; ; NRMSE=0.275)
['GS_F'] for F => (MAE=0.147; RMSE=0.228; ; NRMSE=0.268)
['SC_F'] for F => (MAE=0.147; RMSE=0.228; ; NRMSE=0.268)
['VIX'] for F => (MAE=0.147; RMSE=0.228; ; NRMSE=0.268)
['PCR'] for F => (MAE=0.147; RMSE=0.228; ; NRMSE=0.268)
['GS_F', 'SC_F'] for F => (MAE=0.149; RMSE=0.216; ; NRMSE=0.254)
['GS_F', 'VIX'] for F => (MAE=0.146; RMSE=0.214; ; NRMSE=0.252)
['GS_F', 'PCR'] for F => (MAE=0.145; RMSE=0.213; ; NRMSE=0.250)
['SC_F', 'VIX'] for F => (MAE=0.145; RMSE=0.212; ; NRMSE=0.249)
['SC_F', 'PCR'] for F => (MAE=0.146; RMSE=0.214; ; NRMSE=0.251)
['VIX', 'PCR'] for F => (MAE=0.144; RMSE=0.210; ; NRMSE=0.247)
['GS_F', 'SC_F', 'VIX'] for F => (MAE=0.143; RMSE=0.230; ; NRMSE=0.270)
['GS_F', 'SC_F', 'PCR'] for F => (MAE=0.145; RMSE=0.234; ; NRMSE=0.275)
['SC_F', 'VIX', 'PCR'] for F => (MAE=0.145; RMSE=0.234; ; NRMSE=0.275)
['SC_F', 'VIX', 'PCR'] for F => (MAE=0.145; RMSE=0.234; ; NRMSE=0.275)
['GS_F', 'SC_F', 'VIX', 'PCR'] for F => (MAE=0.132; R